# Lab: Convolutional Neural Networks for Satellite Image Classification

**Instructor:** Muhammad Sayed  
**Semester:** Spring 2026

---

### Learning Objectives
* A custom **Convolutional Neural Network** architecture is designed and implemented using PyTorch.
* **High-resolution satellite imagery** is mapped from fine-grained categories to broad macro-classes.
* **Classification model generalization** is evaluated through systematic **hyperparameter tuning**.
* Classification errors are analyzed by **tracing misclassified macro-classes back to their original micro-class spectral signatures**.
* Results and experimental trials are serialized utilizing Pydantic models for **standardized** evaluation.


In [1]:
# !pip install kagglehub
import kagglehub

# Download latest version
path = kagglehub.dataset_download("aqibrehmanpirzada/nwpuresisc45")

print("Path to dataset files:", path)

c:\Users\Rufai\rssi_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\Rufai\.cache\kagglehub\datasets\aqibrehmanpirzada\nwpuresisc45\versions\1


In [32]:
import os
import json
import pickle
from enum import Enum
import random
from typing import List, Dict, Any
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.datasets import ImageFolder
from torchvision import transforms
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from pydantic import BaseModel
import warnings

warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


### Data Engineering and Macro-Class Mapping
The required NWPU-RESISC45 dataset can be downloaded from [Kaggle](https://www.kaggle.com/datasets/aqibrehmanpirzada/nwpuresisc45). It consists of forty-five fine-grained classes. These are algorithmically mapped to four broad macro-classes. Cloud cover is categorized as unknown and must be filtered out during dataset initialization.

In [3]:
class LandCover(Enum):
    UNKNOWN = 0
    GREENERY = 1
    SAND = 2
    WATER = 3
    CEMENT_INFRASTRUCTURE = 4

macro_class_mapping = {
    'chaparral': LandCover.GREENERY, 
    'circular_farmland': LandCover.GREENERY, 
    'forest': LandCover.GREENERY,
    'golf_course': LandCover.GREENERY, 
    'meadow': LandCover.GREENERY, 
    'rectangular_farmland': LandCover.GREENERY, 
    'terrace': LandCover.GREENERY,

    'beach': LandCover.SAND, 
    'desert': LandCover.SAND, 
    'mountain': LandCover.SAND, 
    'island': LandCover.SAND,

    'harbor': LandCover.WATER, 
    'lake': LandCover.WATER, 
    'river': LandCover.WATER, 
    'sea_ice': LandCover.WATER, 
    'wetland': LandCover.WATER, 
    'ship': LandCover.WATER, 
    'snowberg': LandCover.WATER,

    'airport': LandCover.CEMENT_INFRASTRUCTURE, 
    'bridge': LandCover.CEMENT_INFRASTRUCTURE, 
    'church': LandCover.CEMENT_INFRASTRUCTURE, 
    'commercial_area': LandCover.CEMENT_INFRASTRUCTURE,
    'dense_residential': LandCover.CEMENT_INFRASTRUCTURE, 
    'freeway': LandCover.CEMENT_INFRASTRUCTURE, 
    'industrial_area': LandCover.CEMENT_INFRASTRUCTURE,
    'intersection': LandCover.CEMENT_INFRASTRUCTURE, 
    'medium_residential': LandCover.CEMENT_INFRASTRUCTURE, 
    'mobile_home_park': LandCover.CEMENT_INFRASTRUCTURE,
    'overpass': LandCover.CEMENT_INFRASTRUCTURE, 
    'palace': LandCover.CEMENT_INFRASTRUCTURE, 
    'parking_lot': LandCover.CEMENT_INFRASTRUCTURE, 
    'railway': LandCover.CEMENT_INFRASTRUCTURE,
    'railway_station': LandCover.CEMENT_INFRASTRUCTURE, 
    'roundabout': LandCover.CEMENT_INFRASTRUCTURE, 
    'runway': LandCover.CEMENT_INFRASTRUCTURE,
    'sparse_residential': LandCover.CEMENT_INFRASTRUCTURE, 
    'storage_tank': LandCover.CEMENT_INFRASTRUCTURE, 
    'thermal_power_station': LandCover.CEMENT_INFRASTRUCTURE,
    'baseball_diamond': LandCover.CEMENT_INFRASTRUCTURE, 
    'basketball_court': LandCover.CEMENT_INFRASTRUCTURE, 
    'ground_track_field': LandCover.CEMENT_INFRASTRUCTURE,
    'stadium': LandCover.CEMENT_INFRASTRUCTURE, 
    'tennis_court': LandCover.CEMENT_INFRASTRUCTURE, 
    'airplane': LandCover.CEMENT_INFRASTRUCTURE,

    'cloud': LandCover.UNKNOWN
}

# 0-based label mapping for CrossEntropyLoss
# Greenery: 0, Sand: 1, Water: 2, Cement/Infrastructure: 3
ENUM_TO_IDX = {
    LandCover.GREENERY: 0,
    LandCover.SAND: 1,
    LandCover.WATER: 2,
    LandCover.CEMENT_INFRASTRUCTURE: 3,
}

CLASS_NAMES = ['GREENERY', 'SAND', 'WATER', 'CEMENT_INFRASTRUCTURE']
NUM_CLASSES = len(CLASS_NAMES)

print("Macro class mapping defined with", NUM_CLASSES, "classes:", CLASS_NAMES)

Macro class mapping defined with 4 classes: ['GREENERY', 'SAND', 'WATER', 'CEMENT_INFRASTRUCTURE']


### Dataset Initialization and Loading
The provided dataset directory is natively partitioned into training and testing subsets. Images must be loaded and mapped to their respective macro-classes, ensuring instances mapped to the `UNKNOWN` category are explicitly excluded. The testing subset is to be preserved strictly for final evaluation. To facilitate hyperparameter tuning without data leakage, the training subset must be further partitioned to generate an independent validation subset, or a cross-validation strategy must be implemented.

In [4]:
# ── auto-detect correct class directory (handles extra nested folders) ──────
def find_class_dir(base):
    """Walk down until we find a folder containing multiple class subfolders."""
    current = base
    while True:
        subdirs = [d for d in os.listdir(current)
                   if os.path.isdir(os.path.join(current, d))]
        if len(subdirs) == 1:
            current = os.path.join(current, subdirs[0])
        else:
            return current

dataset_path = os.path.join(path, "Dataset")
path_to_train_dir = find_class_dir(os.path.join(dataset_path, "train"))
path_to_test_dir  = find_class_dir(os.path.join(dataset_path, "test"))
print("Dataset path:", dataset_path)
print("Path to training directory:", path_to_train_dir)
print("Path to testing directory:",  path_to_test_dir)

IMG_SIZE = 64

# ── transform pipelines ────────────────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

Dataset path: C:\Users\Rufai\.cache\kagglehub\datasets\aqibrehmanpirzada\nwpuresisc45\versions\1\Dataset
Path to training directory: C:\Users\Rufai\.cache\kagglehub\datasets\aqibrehmanpirzada\nwpuresisc45\versions\1\Dataset\train\train
Path to testing directory: C:\Users\Rufai\.cache\kagglehub\datasets\aqibrehmanpirzada\nwpuresisc45\versions\1\Dataset\test\test


In [5]:
# ── helper: filter UNKNOWN and remap to 0-based macro labels ───────────────
def build_macro_subset(dataset):
    """
    Iterates over an ImageFolder dataset and:
      - Skips any sample whose micro-class maps to UNKNOWN.
      - Remaps valid samples to 0-based macro-class labels.
    Returns:
      valid_indices  : list[int]  - sample positions to keep
      mapped_targets : list[int]  - 0-based macro label per valid sample
      micro_names    : list[str]  - original micro-class name per valid sample
    """
    valid_indices, mapped_targets, micro_names = [], [], []
    for idx, (_, orig_label) in enumerate(dataset.samples):
        micro_name = dataset.classes[orig_label]
        macro_enum = macro_class_mapping.get(micro_name)
        if macro_enum is None or macro_enum == LandCover.UNKNOWN:
            continue
        valid_indices.append(idx)
        mapped_targets.append(ENUM_TO_IDX[macro_enum])
        micro_names.append(micro_name)
    return valid_indices, mapped_targets, micro_names


# ── MacroSubset: fixes IndexError by returning remapped macro labels ────────
# Using a custom Dataset instead of Subset ensures labels are always
# the remapped 0-based macro labels (0-3), not the original 0-44 labels.
class MacroSubset(Dataset):
    def __init__(self, dataset, indices, labels):
        self.dataset = dataset
        self.indices = indices
        self.labels  = labels

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        image, _ = self.dataset[self.indices[i]]  # ignore original label
        return image, self.labels[i]              # return macro label instead


# ── load raw ImageFolders ───────────────────────────────────────────────────
raw_train      = ImageFolder(root=path_to_train_dir, transform=transform)
raw_train_eval = ImageFolder(root=path_to_train_dir, transform=eval_transform)
raw_test       = ImageFolder(root=path_to_test_dir,  transform=eval_transform)

train_indices, train_labels, train_micros = build_macro_subset(raw_train)
test_indices,  test_labels,  test_micros  = build_macro_subset(raw_test)

print(f"Total raw train samples        : {len(raw_train)}")
print(f"Valid train samples (no cloud) : {len(train_indices)}")
print(f"Total raw test samples         : {len(raw_test)}")
print(f"Valid test samples  (no cloud) : {len(test_indices)}")

# ── train / val split (80 / 20, no leakage) ───────────────────────────────
shuffled   = train_indices[:]
random.shuffle(shuffled)
split      = int(0.8 * len(shuffled))
tr_indices = shuffled[:split]
va_indices = shuffled[split:]

idx_to_macro = {orig_idx: macro_lbl
                for orig_idx, macro_lbl in zip(train_indices, train_labels)}
tr_labels = [idx_to_macro[i] for i in tr_indices]
va_labels = [idx_to_macro[i] for i in va_indices]

train_subset = MacroSubset(raw_train,      tr_indices,   tr_labels)
val_subset   = MacroSubset(raw_train_eval, va_indices,   va_labels)
test_subset  = MacroSubset(raw_test,       test_indices, test_labels)

# ── DataLoaders ────────────────────────────────────────────────────────────
# NUM_WORKERS=0 avoids Windows/Jupyter multiprocessing deadlock
BATCH_SIZE  = 32
NUM_WORKERS = 0

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_subset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_subset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS)

print(f"\nTrain samples : {len(train_subset)}")
print(f"Val   samples : {len(val_subset)}")
print(f"Test  samples : {len(test_subset)}")

# sanity check - labels must only be 0,1,2,3
images, labels = next(iter(train_loader))
print(f"\nBatch shape   : {images.shape}")
print(f"Unique labels : {labels.unique().tolist()}  <- should only contain 0,1,2,3")

Total raw train samples        : 27000
Valid train samples (no cloud) : 26400
Total raw test samples         : 4500
Valid test samples  (no cloud) : 4400

Train samples : 21120
Val   samples : 5280
Test  samples : 4400

Batch shape   : torch.Size([32, 3, 64, 64])
Unique labels : [0, 1, 2, 3]  <- should only contain 0,1,2,3


### Convolutional Neural Network Architecture
A custom CNN class is defined. Spatial dimensions must be carefully tracked between convolutional outputs and fully connected layer inputs.

As a reference model, the LeNet-5 architecture illustrates the foundational principles of convolutional networks. The input tensor is sequentially processed through alternating convolutional and pooling layers to extract spatial features and reduce dimensionality. The resulting feature maps are subsequently flattened and passed through fully connected layers to yield the final classification probabilities.

**Spatial dimension tracking** (input = 3 × 64 × 64):

| Layer | Output shape |
|---|---|
| Conv1 (3→32, 3×3, pad=1) + BN + ReLU | 32 × 64 × 64 |
| MaxPool 2×2 | 32 × 32 × 32 |
| Conv2 (32→64, 3×3, pad=1) + BN + ReLU | 64 × 32 × 32 |
| MaxPool 2×2 | 64 × 16 × 16 |
| Conv3 (64→128, 3×3, pad=1) + BN + ReLU | 128 × 16 × 16 |
| MaxPool 2×2 | 128 × 8 × 8 |
| Flatten | 8192 |
| FC1 (8192→256) + ReLU + Dropout | 256 |
| FC2 (256→128) + ReLU + Dropout | 128 |
| FC3 (128→4) | 4 (logits) |


In [6]:
class SatelliteCNN(nn.Module):

    def __init__(self, num_classes: int = 4, dropout: float = 0.5):
        super(SatelliteCNN, self).__init__()

        # ── convolutional blocks ──────────────────────────────────────────
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)          # 64 -> 32
        )
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)          # 32 -> 16
        )
        self.conv_block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)          # 16 -> 8
        )

        # ── fully connected head ──────────────────────────────────────────
        # Flattened size = 128 channels * 8 * 8 = 8192
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.classifier(x)
        return x


# sanity check: output shape should be [2, 4]
dummy = torch.zeros(2, 3, IMG_SIZE, IMG_SIZE)
out   = SatelliteCNN(num_classes=NUM_CLASSES)(dummy)
print(f"Output shape: {out.shape}  <- should be [2, 4]")

Output shape: torch.Size([2, 4])  <- should be [2, 4]


### Model Training
The loss function and optimizer are initialized. The training loop is executed, updating model weights while validating performance on the held-out validation set to monitor for overfitting.

In [7]:
def train_model(model, train_loader, val_loader, criterion, optimizer,
                num_epochs=10, device=device, scheduler=None):
    """
    Trains the model for num_epochs and returns the best model
    (checkpoint selected by lowest validation loss).
    """
    best_val_loss   = float('inf')
    best_state_dict = None

    for epoch in range(1, num_epochs + 1):

        # ── training phase ───────────────────────────────────────────────
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            preds         = outputs.argmax(dim=1)
            correct      += (preds == labels).sum().item()
            total        += images.size(0)

        train_loss = running_loss / total
        train_acc  = correct / total

        # ── validation phase ─────────────────────────────────────────────
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs        = model(images)
                loss           = criterion(outputs, labels)
                val_loss    += loss.item() * images.size(0)
                preds        = outputs.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_total   += images.size(0)

        val_loss /= val_total
        val_acc   = val_correct / val_total

        # save best checkpoint
        if val_loss < best_val_loss:
            best_val_loss   = val_loss
            best_state_dict = {k: v.cpu().clone()
                               for k, v in model.state_dict().items()}

        # step the LR scheduler if provided
        if scheduler is not None:
            scheduler.step()

        print(f"Epoch {epoch:>2}/{num_epochs}  "
              f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  "
              f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}")

    # restore best weights before returning
    model.load_state_dict(best_state_dict)
    print("\nBest model restored (lowest val loss).")
    return model


# ── instantiate model, loss, optimizer, scheduler ─────────────────────────
model     = SatelliteCNN(num_classes=NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

# ── train ──────────────────────────────────────────────────────────────────
model = train_model(model, train_loader, val_loader, criterion, optimizer,
                    num_epochs=10, device=device, scheduler=scheduler)

Epoch  1/10  Train Loss: 0.9347  Train Acc: 0.6470  Val Loss: 0.8194  Val Acc: 0.7106
Epoch  2/10  Train Loss: 0.7884  Train Acc: 0.7083  Val Loss: 0.6865  Val Acc: 0.7470
Epoch  3/10  Train Loss: 0.7476  Train Acc: 0.7261  Val Loss: 0.6861  Val Acc: 0.7383
Epoch  4/10  Train Loss: 0.6861  Train Acc: 0.7441  Val Loss: 0.6053  Val Acc: 0.7748
Epoch  5/10  Train Loss: 0.6570  Train Acc: 0.7567  Val Loss: 0.5728  Val Acc: 0.7790
Epoch  6/10  Train Loss: 0.6337  Train Acc: 0.7661  Val Loss: 0.6253  Val Acc: 0.7831
Epoch  7/10  Train Loss: 0.5839  Train Acc: 0.7798  Val Loss: 0.5023  Val Acc: 0.8119
Epoch  8/10  Train Loss: 0.5637  Train Acc: 0.7904  Val Loss: 0.5161  Val Acc: 0.8261
Epoch  9/10  Train Loss: 0.5425  Train Acc: 0.8005  Val Loss: 0.4948  Val Acc: 0.8191
Epoch 10/10  Train Loss: 0.5191  Train Acc: 0.8069  Val Loss: 0.4570  Val Acc: 0.8271

Best model restored (lowest val loss).


### Quantitative Evaluation
The trained model is evaluated against the test subset. Overall Accuracy, Macro-Averaged F1 Scores, and the Confusion Matrix are computed.

In [36]:
def evaluate_model(model, loader, device=device):
    """
    Runs inference on all batches in loader.
    Returns (all_preds, all_labels) as plain Python lists.
    Exposed at module level so Person 2 can reuse it in tuning trials.
    """
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images  = images.to(device)
            outputs = model(images)
            preds   = outputs.argmax(dim=1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels.tolist())

    return all_preds, all_labels



In [ ]:

# ── run evaluation on held-out test set ───────────────────────────────────
test_preds, test_labels_list = evaluate_model(model, test_loader)

overall_acc  = accuracy_score(test_labels_list, test_preds)
f1_macro     = f1_score(test_labels_list, test_preds, average='macro')
f1_per_class = f1_score(test_labels_list, test_preds, average=None)
cm           = confusion_matrix(test_labels_list, test_preds)

print(f"Overall Accuracy : {overall_acc:.4f}")
print(f"Macro F1 Score   : {f1_macro:.4f}")
print("\nPer-class F1 Scores:")
for name, score in zip(CLASS_NAMES, f1_per_class):
    print(f"  {name:<30}: {score:.4f}")
print("\nConfusion Matrix (rows = true, cols = predicted):")
print(f"Labels: {CLASS_NAMES}")
print(cm)

### Save Handoff for Person 2
Saves the trained model and all dataset variables into `person1_handoff.pkl`. Share this file with Person 2 so they do not need to re-run training.

In [ ]:
data = {
    'model_state_dict': model.state_dict(),
    'tr_indices':   tr_indices,
    'va_indices':   va_indices,
    'test_indices': test_indices,
    'tr_labels':    tr_labels,
    'va_labels':    va_labels,
    'test_labels':  test_labels,
    'test_micros':  test_micros,
    'TRAIN_DIR':    path_to_train_dir,
    'TEST_DIR':     path_to_test_dir,
    'BATCH_SIZE':   BATCH_SIZE,
    'NUM_CLASSES':  NUM_CLASSES,
    'IMG_SIZE':     IMG_SIZE,
    'CLASS_NAMES':  CLASS_NAMES,
}

with open('data.pkl', 'wb') as f:
    pickle.dump(data, f)

print("Saved data.pkl")
print(f"  Train samples : {len(tr_indices)}")
print(f"  Val   samples : {len(va_indices)}")
print(f"  Test  samples : {len(test_indices)}")

In [ ]:
# ── load saved data file ────────────────────────────────────────────────
with open('data.pkl', 'rb') as f:
    data = pickle.load(f)

TRAIN_DIR   = data['TRAIN_DIR']
TEST_DIR    = data['TEST_DIR']
BATCH_SIZE  = data['BATCH_SIZE']
NUM_CLASSES = data['NUM_CLASSES']
IMG_SIZE    = data['IMG_SIZE']
CLASS_NAMES = data['CLASS_NAMES']
tr_indices  = data['tr_indices']
va_indices  = data['va_indices']
test_indices= data['test_indices']
tr_labels   = data['tr_labels']
va_labels   = data['va_labels']
test_labels = data['test_labels']
test_micros = data['test_micros']

# ── rebuild transforms and datasets ───────────────────────────────────────
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
eval_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

raw_train      = ImageFolder(root=TRAIN_DIR, transform=train_transforms)
raw_train_eval = ImageFolder(root=TRAIN_DIR, transform=eval_transforms)
raw_test       = ImageFolder(root=TEST_DIR,  transform=eval_transforms)

class MacroSubset(Dataset):
    def __init__(self, dataset, indices, labels):
        self.dataset = dataset
        self.indices = indices
        self.labels  = labels
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        image, _ = self.dataset[self.indices[i]]
        return image, self.labels[i]

train_subset = MacroSubset(raw_train,      tr_indices,   tr_labels)
val_subset   = MacroSubset(raw_train_eval, va_indices,   va_labels)
test_subset  = MacroSubset(raw_test,       test_indices, test_labels)

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_subset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_subset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# ── restore trained model ──────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from torch import nn
class SatelliteCNN(nn.Module):
    def __init__(self, num_classes=4, dropout=0.5):
        super().__init__()
        self.conv_block1 = nn.Sequential(nn.Conv2d(3,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2,2))
        self.conv_block2 = nn.Sequential(nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2,2))
        self.conv_block3 = nn.Sequential(nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2,2))
        self.classifier  = nn.Sequential(nn.Flatten(), nn.Linear(128*8*8,256), nn.ReLU(inplace=True), nn.Dropout(dropout), nn.Linear(256,128), nn.ReLU(inplace=True), nn.Dropout(dropout), nn.Linear(128,num_classes))
    def forward(self, x):
        return self.classifier(self.conv_block3(self.conv_block2(self.conv_block1(x))))

model = SatelliteCNN(num_classes=NUM_CLASSES).to(device)
model.load_state_dict(data['model_state_dict'])
model.eval()

print("✓ Data loaded successfully")
print(f"  Device    : {device}")
print(f"  Classes   : {CLASS_NAMES}")
print(f"  Test size : {len(test_subset)}")

# # print model evaluation results to verify we loaded everything correctly
# test_preds, test_labels_list = evaluate_model(model, test_loader, device=device)
# overall_acc  = accuracy_score(test_labels_list, test_preds)
# print(f"\nTest Accuracy after loading: {overall_acc:.4f}  <- should match previous accuracy")

✓ Data loaded successfully
  Device    : cpu
  Classes   : ['GREENERY', 'SAND', 'WATER', 'CEMENT_INFRASTRUCTURE']
  Test size : 4400

Test Accuracy after loading: 0.8323  <- should match previous accuracy


### Hyperparameter Tuning Analysis
Observations regarding hyperparameter tuning are required. Multiple trials must be recorded, documenting the chosen parameters, resultant accuracy, class-specific F-scores, and drawn conclusions. These records are to be serialized using the provided Pydantic schema in the submission cell.


In [10]:
# Hyperparameter tuning execution is to be implemented here.
# Utilizing a grid search strategy (e.g., via itertools.product or sklearn.model_selection.ParameterGrid) is suggested to systematically explore combinations of learning rates, batch sizes, optimizers, and epochs.


### Error Analysis and Confusion Patterns
The macro-class exhibiting the lowest accuracy must be identified based on the computed F-scores and Confusion Matrix. Subsequently, the original, fine-grained micro-classes constituting the misclassified instances within this macro-class are to be examined. Patterns explaining why the convolutional neural network struggles to differentiate these specific land cover categories must be formulated and reported in the submission payload.


In [11]:
# Error analysis execution is to be implemented here.
# A confusion matrix or cross-tabulation mapping the true original micro-classes against the predicted macro-classes for the least accurate macro-class must be generated here to facilitate pattern recognition.


### Pydantic Models and Submission Export
The `pydantic` library is utilized to ensure structural integrity of the final submission. The models below define the exact schema expected for automated grading. The variables are to be populated with the experimental findings, and the cell executed to write the JSON output.

In [ ]:
class TrialRecord(BaseModel):
    hyperparams: Dict[str, Any]
    accuracy: float
    F_score_class: Dict[str, float]
    observations: List[str]
    conclusions: List[str]

class LabSubmission(BaseModel):
    student_ids: List[str]
    trials: List[TrialRecord]
    micro_class_error_analysis: str

# The records are populated here based on experimental iterations.
trial_one = TrialRecord(
    hyperparams={"learning_rate": 0.001, "batch_size": 32, "optimizer": "Adam", "epochs": 10},
    accuracy=0.0,
    F_score_class={"GREENERY": 0.0, "SAND": 0.0, "WATER": 0.0, "CEMENT_INFRASTRUCTURE": 0.0},
    observations=["Initial training phase completed.", "Validation loss fluctuated."],
    conclusions=["A lower learning rate may be required.", "Further regularization is recommended."]
)

# Additional trials are to be added to the list below.
submission = LabSubmission(
    student_ids=["9202550", "9000001"],  # must be strings
    trials=[trial_one],
    micro_class_error_analysis="Insert the detailed analysis required by Question D here, discussing the specific original micro-classes that were misclassified within the worst-performing macro-class."
)

# The Pydantic model is serialized to JSON and saved for grading.
output_json = submission.model_dump_json(indent=4)
print(output_json)

with open("student_submission.json", "w") as f:
    f.write(output_json)


{
    "student_ids": [
        "9000000",
        "9000001"
    ],
    "trials": [
        {
            "hyperparams": {
                "learning_rate": 0.001,
                "batch_size": 32,
                "optimizer": "Adam",
                "epochs": 10
            },
            "accuracy": 0.0,
            "F_score_class": {
                "GREENERY": 0.0,
                "SAND": 0.0,
                "WATER": 0.0,
                "CEMENT_INFRASTRUCTURE": 0.0
            },
            "observations": [
                "Initial training phase completed.",
                "Validation loss fluctuated."
            ],
            "conclusions": [
                "A lower learning rate may be required.",
                "Further regularization is recommended."
            ]
        }
    ],
    "micro_class_error_analysis": "Insert the detailed analysis required by Question D here, discussing the specific original micro-classes that were misclassified within the worst-performing 

### Submission Protocol

The final deliverable **must be a zipped archive** submitted via Google Classroom. 

* The archive must contain the executed Jupyter Notebook and the generated `student_submission.json` file.
* The zipped file must be explicitly named utilizing the display name of the submitter exactly as it appears on Google Classroom.
* **Failure to adhere to these archival and naming conventions disrupts automated grading pipelines and will result in direct grade penalties.**

---

### Grading Rubric (Total: Ten Marks)

The requirement submission is evaluated based on the following criteria:

* **Data Engineering and Macro-Class Mapping (Two Marks)**
  The dataset is correctly loaded, successfully mapped to the defined Enum macro-classes, and securely partitioned into distinct training, validation, and testing subsets without data leakage.

* **Architecture and Training Execution (Two Marks)**
  A functional Convolutional Neural Network is constructed with appropriate spatial dimension tracking, and the training loop successfully computes gradients and updates model weights.

* **Hyperparameter Tuning (Two Marks)**
  Systematic hyperparameter tuning is executed, and multiple experimental trials are accurately serialized within the final JSON payload.

* **Error Analysis and Custom Confusion Matrix (Two Marks)**
  A custom confusion matrix mapping true micro-classes against predicted macro-classes is successfully generated, and systematic misclassification patterns for the least accurate class are logically analyzed in the written submission.

* **Quantitative Evaluation and Peer Comparison (Two Marks)**
  The macro-averaged F-score from the final experimental trial in the submitted list is explicitly reported and analytically compared against peer performances to identify relative architectural strengths and areas for optimization.
